# RAPID2 Tutorial on MAAP DPS

This notebook recreates the [RAPID2 Tutorial](https://github.com/c-h-david/rapid2/blob/main/TUTORIAL.md) on the MAAP platform.

**Architecture:**
- **Notebook (local):** Lightweight pre-processing — download runoff data, transform to external inflow, generate cold start
- **DPS Job (remote):** Compute-heavy Muskingum routing via `rapid2`

**Prerequisites:**
- Algorithm `rapid2` (version `maap`) is already registered on MAAP DPS
- Earthdata credentials are configured (auto-provided on MAAP)
- Basin: `pfaf_74` (Mississippi River Basin)

## Setup

In [ ]:
!pip install -q rapid2

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path
from urllib.parse import urlparse
import boto3
from maap.maap import MAAP

maap = MAAP(maap_host="api.maap-project.org")

BASIN = "pfaf_74"
WORK_DIR = Path("rapid2_tutorial")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- MAAP user info ---
username = maap.profile.account_info()["username"]
print(f"Username: {username}")
print(f"Basin: {BASIN}")

# --- Workspace bucket credentials (reused in Steps 5, 6, 7) ---
print("Fetching workspace bucket credentials...")
ws_creds = maap.aws.workspace_bucket_credentials()
creds = ws_creds["credentials"]
s3_ws = boto3.client(
    "s3",
    aws_access_key_id=creds["aws_access_key_id"],
    aws_secret_access_key=creds["aws_secret_access_key"],
    aws_session_token=creds["aws_session_token"],
)
BUCKET = "maap-ops-workspace"
S3_PREFIX = f"{username}/rapid2/tutorial/{BASIN}"
print(f"  Workspace bucket: s3://{BUCKET}/{S3_PREFIX}")
print(f"  Authorized paths: {[p['uri'] for p in ws_creds['authorized_s3_paths']]}")

print(f"\nWorking directory: {WORK_DIR}")
print("Setup complete.")

## Step 1: Download Static Files from Zenodo

Download connectivity, coordinates, and coupling files for `pfaf_74` from Zenodo DOI [10.5281/zenodo.8248069](https://doi.org/10.5281/zenodo.8248069).

In [ ]:
ZENODO_RECORD = "10013744"
ZENODO_BASE = f"https://zenodo.org/records/{ZENODO_RECORD}/files"

ZIP_FILES = [
    "rapid_connect_pfaf_ii.zip",
    "coords_pfaf_ii.zip",
    "rapid_coupling_pfaf_ii_GLDAS.zip",
    "k_pfaf_ii_nrm.zip",
    "x_pfaf_ii_nrm.zip",
    "riv_bas_id_pfaf_ii_topo.zip",
]

print(f"Downloading {len(ZIP_FILES)} ZIP files from Zenodo record {ZENODO_RECORD}...")
for i, zf in enumerate(ZIP_FILES, 1):
    url = f"{ZENODO_BASE}/{zf}?download=1"
    dest = INPUT_DIR / zf
    if not dest.exists():
        print(f"  [{i}/{len(ZIP_FILES)}] Downloading {zf}...")
        urllib.request.urlretrieve(url, dest)
    else:
        print(f"  [{i}/{len(ZIP_FILES)}] Already have {zf}")
    with zipfile.ZipFile(dest, "r") as z:
        z.extractall(INPUT_DIR)

print(f"\nStatic files for {BASIN}:")
for f in sorted(INPUT_DIR.glob(f"*{BASIN}*")):
    print(f"  {f.name}")

## Step 2: Download GLDAS Runoff Data

Downloads GLDAS phase 2.1, VIC model, 3-hourly runoff data for January 2010 directly from GES DISC's S3 bucket using the Hub's built-in AWS credentials (requester-pays). We search NASA CMR via `maap.searchGranule` to discover granule S3 paths, then download with `boto3`.

This approach follows the [external access from Hub tutorial](https://docs.maap-project.org/en/latest/technical_tutorials/access/external_access_from_hub.html) — no Earthdata Login required.

In [ ]:
import netCDF4
import numpy as np

GLDAS_FILE = INPUT_DIR / "GLDAS_2.1_VIC_2010-01.nc4"
GLDAS_RAW_DIR = INPUT_DIR / "gldas_raw"
GLDAS_RAW_DIR.mkdir(exist_ok=True)

# --- 1. Search for GLDAS granules in NASA CMR ---
print("Searching NASA CMR for GLDAS granules...")
results = maap.searchGranule(
    cmr_host="cmr.earthdata.nasa.gov",
    short_name="GLDAS_VIC10_3H",
    version="2.1",
    temporal="2010-01-01T00:00:00Z,2010-01-31T23:59:59Z",
    limit=300,
)
print(f"Found {len(results)} granules")

# --- 2. Download directly from GES DISC S3 (requester-pays, no EDL needed) ---
print("Downloading from GES DISC S3 (requester-pays)...")
os.environ["AWS_REQUEST_PAYER"] = "requester"
s3 = boto3.client("s3")

for i, granule in enumerate(results):
    s3_url = granule.getS3Url()
    if not s3_url:
        continue
    parsed = urlparse(s3_url)
    bucket = parsed.netloc
    key = parsed.path.lstrip("/")
    filename = key.split("/")[-1]
    dest = GLDAS_RAW_DIR / filename
    if not dest.exists():
        s3.download_file(bucket, key, str(dest), ExtraArgs={"RequestPayer": "requester"})
    if (i + 1) % 50 == 0:
        print(f"  Downloaded {i + 1}/{len(results)} granules...")

raw_files = sorted(GLDAS_RAW_DIR.glob("*.nc4"))
print(f"Downloaded {len(raw_files)} files")

# --- 3. Concatenate into single NetCDF ---
print("Concatenating into single NetCDF...")
keep_vars = {"time", "time_bnds", "lon", "lat", "Qs_acc", "Qsb_acc"}

with netCDF4.Dataset(raw_files[0], "r") as src, \
     netCDF4.Dataset(GLDAS_FILE, "w") as dst:
    dst.setncatts({attr: src.getncattr(attr) for attr in src.ncattrs()})
    for name, dim in src.dimensions.items():
        dst.createDimension(name, None if dim.isunlimited() else len(dim))
    for name, var in src.variables.items():
        if name in keep_vars:
            dst.createVariable(name, var.datatype, var.dimensions).setncatts(
                {attr: var.getncattr(attr) for attr in var.ncattrs()}
            )

time_idx = 0
for f in raw_files:
    with netCDF4.Dataset(f, "r") as src, \
         netCDF4.Dataset(GLDAS_FILE, "a") as dst:
        n_time = src.dimensions["time"].size
        for name, var in src.variables.items():
            if name in keep_vars:
                if "time" in var.dimensions:
                    dst.variables[name][time_idx:time_idx + n_time] = var[:]
                else:
                    dst.variables[name][:] = var[:]
        time_idx += n_time

# --- 4. Fix time units (GLDAS 2.1 specific) ---
print("Fixing time units...")
with netCDF4.Dataset(GLDAS_FILE, "a") as ds:
    ds.variables["time"][:] = ds.variables["time"][:] * 60 + 946695600
    ds.variables["time_bnds"][:] = ds.variables["time_bnds"][:] * 60 + 946695600
    ds.variables["time"].units = "second since 1970-01-01 00:00:00 +00:00"

# --- 5. Convert accumulated depth to depth rate ---
with netCDF4.Dataset(GLDAS_FILE, "a") as ds:
    dt = ds.variables["time_bnds"][0, 1] - ds.variables["time_bnds"][0, 0]
    if dt < 100000:
        print(f"Converting accumulations by dt={dt}s")
        ds.variables["Qs_acc"][:] = ds.variables["Qs_acc"][:] / dt
        ds.variables["Qsb_acc"][:] = ds.variables["Qsb_acc"][:] / dt
        ds.variables["Qs_acc"].units = "kg m-2 s-1"
        ds.variables["Qsb_acc"].units = "kg m-2 s-1"

# --- 6. Cleanup raw files ---
print("Cleaning up raw files...")
for f in raw_files:
    f.unlink()
GLDAS_RAW_DIR.rmdir()

print(f"Done: {GLDAS_FILE} ({GLDAS_FILE.stat().st_size / 1024 / 1024:.1f} MB)")

## Step 3: Transform Runoff to External Inflow

Uses `cpllsm` to couple the land surface model output with the river network.

In [ ]:
QEXT_FILE = INPUT_DIR / f"Qext_{BASIN}_GLDAS_2.1_VIC_2010-01.nc4"

print(f"Running cpllsm to transform runoff → external inflow...")
print(f"  Input LSM: {GLDAS_FILE}")
print(f"  Output: {QEXT_FILE}")

!cpllsm \
    --land_surface_model \
    {GLDAS_FILE} \
    --connectivity \
    {INPUT_DIR}/rapid_connect_{BASIN}.csv \
    --coordinates \
    {INPUT_DIR}/coords_{BASIN}.csv \
    --coupling \
    {INPUT_DIR}/rapid_coupling_{BASIN}_GLDAS.csv \
    --external_inflow \
    {QEXT_FILE}

if QEXT_FILE.exists():
    print(f"\nDone: {QEXT_FILE.name} ({QEXT_FILE.stat().st_size / 1024:.1f} KB)")
else:
    print("\nERROR: Output file not created. Check cpllsm output above.")

## Step 4: Generate Cold Start (Zero Initial Discharge)

Uses `zeroqinit` to create an initial condition file with all flows set to zero.

In [ ]:
QINIT_FILE = INPUT_DIR / f"Qinit_{BASIN}_GLDAS_2.1_VIC_2010-01.nc4"

print(f"Running zeroqinit to generate cold start file...")
print(f"  Input: {QEXT_FILE}")
print(f"  Output: {QINIT_FILE}")

!zeroqinit \
    --external_inflow \
    {QEXT_FILE} \
    --initial_outflow \
    {QINIT_FILE}

if QINIT_FILE.exists():
    print(f"\nDone: {QINIT_FILE.name} ({QINIT_FILE.stat().st_size / 1024:.1f} KB)")
else:
    print("\nERROR: Output file not created. Check zeroqinit output above.")

## Step 5: Upload Inputs to MAAP S3

Upload the 6 input files needed by the DPS job to your workspace bucket using credentials from `maap.aws.workspace_bucket_credentials()`.

In [ ]:
DPS_INPUTS = {
    "Qex_ncf": QEXT_FILE,
    "Q00_ncf": QINIT_FILE,
    "con_csv": INPUT_DIR / f"rapid_connect_{BASIN}.csv",
    "kpr_csv": INPUT_DIR / f"k_{BASIN}_nrm.csv",
    "xpr_csv": INPUT_DIR / f"x_{BASIN}_nrm.csv",
    "bas_csv": INPUT_DIR / f"riv_bas_id_{BASIN}_topo.csv",
}

print(f"Uploading {len(DPS_INPUTS)} files to s3://{BUCKET}/{S3_PREFIX}/...")
s3_urls = {}
for key, local_path in DPS_INPUTS.items():
    s3_key = f"{S3_PREFIX}/{local_path.name}"
    s3_ws.upload_file(str(local_path), BUCKET, s3_key)
    s3_url = f"s3://{BUCKET}/{s3_key}"
    s3_urls[key] = s3_url
    print(f"  {key}: {s3_url}")

print("\nAll inputs uploaded.")

## Step 6: Submit DPS Job for Routing

Submit the compute-heavy `rapid2` Muskingum routing as a DPS job. This is the step that benefits from scaling — it iterates over every river reach at every time step.

In [ ]:
IS_dtR = "900"

print("Submitting DPS job...")
print(f"  Algorithm: rapid2 (version: maap)")
print(f"  Queue: maap-dps-worker-8gb")
print(f"  Basin: {BASIN}")
print(f"  Routing time step: {IS_dtR}s")
print(f"  Inputs:")
for key, url in s3_urls.items():
    print(f"    {key}: {url}")

job = maap.submitJob(
    identifier=f"rapid2_tutorial_{BASIN}",
    algo_id="rapid2",
    version="maap",
    queue="maap-dps-worker-8gb",
    username=username,
    Qex_ncf=s3_urls["Qex_ncf"],
    Q00_ncf=s3_urls["Q00_ncf"],
    con_csv=s3_urls["con_csv"],
    kpr_csv=s3_urls["kpr_csv"],
    xpr_csv=s3_urls["xpr_csv"],
    bas_csv=s3_urls["bas_csv"],
    IS_dtR=IS_dtR,
)

print(f"\nJob submitted!")
print(f"  Job ID: {job.id}")
print(f"  Status: {job.status}")

In [ ]:
import time

print("Waiting for job to complete (exponential backoff, may take several minutes)...")
print(f"  Job ID: {job.id}")
t0 = time.time()

job.wait_for_completion()

elapsed = time.time() - t0
print(f"\nJob finished in {elapsed:.0f}s")
print(f"  Final status: {job.status}")

if job.status == "Succeeded":
    print("\nRetrieving outputs...")
    job.retrieve_result()
    print(f"  Found {len(job.outputs)} output URL(s):")
    for url in job.outputs:
        print(f"    {url}")
else:
    print(f"\nJob did not succeed. Status: {job.status}")
    try:
        print(f"  Error: {job.error_details}")
    except:
        pass
    try:
        job.retrieve_result()
        if job.outputs:
            print(f"  Partial outputs: {job.outputs}")
    except:
        pass

## Step 7: Download and Verify Outputs

Download the routing outputs from DPS and run `cmpncf` to compare against gold standard files (if available).

In [ ]:
print("Downloading outputs from S3...")
print(f"  Output URLs from DPS:")
for url in job.outputs:
    print(f"    {url}")

def parse_dps_s3_url(url):
    """Parse DPS S3 URLs which may be path-style with host:port."""
    parsed = urlparse(url)
    host = parsed.netloc
    if "amazonaws.com" in host:
        # Path-style: s3://s3-us-west-2.amazonaws.com:80/bucket/prefix/...
        path_parts = parsed.path.lstrip("/").split("/", 1)
        return path_parts[0], path_parts[1] if len(path_parts) > 1 else ""
    else:
        # Virtual-hosted: s3://bucket/prefix/...
        return host, parsed.path.lstrip("/")

downloaded = []
for output_url in job.outputs:
    if not output_url.startswith("s3://"):
        print(f"\n  Skipping non-S3 URL: {output_url}")
        continue

    bucket, prefix = parse_dps_s3_url(output_url)
    print(f"\n  Listing objects in s3://{bucket}/{prefix}")

    # List all objects under this prefix (DPS output is a directory)
    response = s3_ws.list_objects_v2(Bucket=bucket, Prefix=prefix)
    objects = response.get("Contents", [])

    if not objects:
        # Maybe it's a single file, not a prefix
        print(f"    No objects found under prefix, trying as direct key...")
        filename = prefix.split("/")[-1]
        if filename:
            local_out = OUTPUT_DIR / filename
            s3_ws.download_file(bucket, prefix, str(local_out))
            downloaded.append(local_out)
            print(f"    Downloaded: {local_out} ({local_out.stat().st_size / 1024:.1f} KB)")
        continue

    print(f"    Found {len(objects)} object(s):")
    for obj in objects:
        key = obj["Key"]
        filename = key.split("/")[-1]
        if not filename:
            continue
        local_out = OUTPUT_DIR / filename
        print(f"    Downloading: {filename} ({obj['Size'] / 1024:.1f} KB)")
        s3_ws.download_file(bucket, key, str(local_out))
        downloaded.append(local_out)

print(f"\nDownloaded {len(downloaded)} file(s) to {OUTPUT_DIR}/")

In [ ]:
print("=== Output Verification ===\n")

QOUT_FILE = OUTPUT_DIR / "Qout_maap.nc4"
QFINAL_FILE = OUTPUT_DIR / "Qfinal_maap.nc4"

for label, fpath in [("Routing output (Qout)", QOUT_FILE), ("Final state (Qfinal)", QFINAL_FILE)]:
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        print(f"  {label}: {fpath.name} ({size_kb:.1f} KB)")
    else:
        print(f"  {label}: NOT FOUND — check DPS job logs")

print(f"\nAll output files in {OUTPUT_DIR}/:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## Multi-Basin Scaling

RAPID2's architecture makes multi-basin processing straightforward on MAAP DPS:

| Step | Scope | Notes |
|------|-------|-------|
| Static files (Step 1) | **Run once** | Zenodo ZIPs contain all 61 PFAF level-2 basins |
| GLDAS runoff (Step 2) | **Run once** | Global gridded data, basin-independent |
| `cpllsm` (Step 3) | **Per basin** | Couples global runoff to basin-specific river network |
| `zeroqinit` (Step 4) | **Per basin** | Cold start from basin-specific external inflow |
| Upload to S3 (Step 5) | **Per basin** | Each basin gets its own S3 prefix |
| DPS routing job (Step 6) | **Per basin, in parallel** | Independent jobs can run simultaneously |

**Available basins** (61 globally, from Zenodo [10.5281/zenodo.8248069](https://doi.org/10.5281/zenodo.8248069)):
- `pfaf_74` — Mississippi River (North America) ← used above
- `pfaf_76` — Columbia River (Pacific Northwest)
- `pfaf_62` — Amazon River (South America)
- `pfaf_22` — Danube River (Europe)
- `pfaf_12` — Congo River (Africa)
- ... and 56 more

The cell below runs the full pipeline for 3 additional basins, submits DPS jobs in parallel, then downloads and visualizes the outputs.

In [ ]:
import subprocess
import time

BASINS = ["pfaf_76", "pfaf_62", "pfaf_22"]  # Columbia, Amazon, Danube
BASIN_NAMES = {"pfaf_76": "Columbia River", "pfaf_62": "Amazon River", "pfaf_22": "Danube River"}

# ============================================================
# Phase 1: Local preprocessing (cpllsm + zeroqinit + upload)
# ============================================================
print("=" * 60)
print("PHASE 1: Local preprocessing for each basin")
print("=" * 60)

basin_s3_urls = {}

for basin in BASINS:
    print(f"\n{'—' * 40}")
    print(f"Basin: {basin} ({BASIN_NAMES[basin]})")
    print(f"{'—' * 40}")

    # --- cpllsm: transform runoff → external inflow ---
    qext_file = INPUT_DIR / f"Qext_{basin}_GLDAS_2.1_VIC_2010-01.nc4"
    print(f"  Running cpllsm...")
    result = subprocess.run(
        [
            "cpllsm",
            "--land_surface_model", str(GLDAS_FILE),
            "--connectivity", str(INPUT_DIR / f"rapid_connect_{basin}.csv"),
            "--coordinates", str(INPUT_DIR / f"coords_{basin}.csv"),
            "--coupling", str(INPUT_DIR / f"rapid_coupling_{basin}_GLDAS.csv"),
            "--external_inflow", str(qext_file),
        ],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"  ERROR cpllsm: {result.stderr}")
        continue
    print(f"    → {qext_file.name} ({qext_file.stat().st_size / 1024:.1f} KB)")

    # --- zeroqinit: generate cold start ---
    qinit_file = INPUT_DIR / f"Qinit_{basin}_GLDAS_2.1_VIC_2010-01.nc4"
    print(f"  Running zeroqinit...")
    result = subprocess.run(
        [
            "zeroqinit",
            "--external_inflow", str(qext_file),
            "--initial_outflow", str(qinit_file),
        ],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"  ERROR zeroqinit: {result.stderr}")
        continue
    print(f"    → {qinit_file.name} ({qinit_file.stat().st_size / 1024:.1f} KB)")

    # --- Upload to S3 ---
    print(f"  Uploading to S3...")
    prefix = f"{username}/rapid2/tutorial/{basin}"
    inputs = {
        "Qex_ncf": qext_file,
        "Q00_ncf": qinit_file,
        "con_csv": INPUT_DIR / f"rapid_connect_{basin}.csv",
        "kpr_csv": INPUT_DIR / f"k_{basin}_nrm.csv",
        "xpr_csv": INPUT_DIR / f"x_{basin}_nrm.csv",
        "bas_csv": INPUT_DIR / f"riv_bas_id_{basin}_topo.csv",
    }
    urls = {}
    for key, local_path in inputs.items():
        s3_key = f"{prefix}/{local_path.name}"
        s3_ws.upload_file(str(local_path), BUCKET, s3_key)
        urls[key] = f"s3://{BUCKET}/{s3_key}"
    basin_s3_urls[basin] = urls
    print(f"    Uploaded 6 files to s3://{BUCKET}/{prefix}/")

print(f"\n\nPreprocessing complete for {len(basin_s3_urls)} basin(s).")

# ============================================================
# Phase 2: Submit DPS jobs in parallel
# ============================================================
print("\n" + "=" * 60)
print("PHASE 2: Submitting DPS jobs")
print("=" * 60)

jobs = {}
for basin, urls in basin_s3_urls.items():
    job = maap.submitJob(
        identifier=f"rapid2_tutorial_{basin}",
        algo_id="rapid2",
        version="maap",
        queue="maap-dps-worker-8gb",
        username=username,
        Qex_ncf=urls["Qex_ncf"],
        Q00_ncf=urls["Q00_ncf"],
        con_csv=urls["con_csv"],
        kpr_csv=urls["kpr_csv"],
        xpr_csv=urls["xpr_csv"],
        bas_csv=urls["bas_csv"],
        IS_dtR="900",
    )
    jobs[basin] = job
    print(f"  {basin} ({BASIN_NAMES[basin]}): Job ID = {job.id}")

# ============================================================
# Phase 3: Wait for all jobs
# ============================================================
print("\n" + "=" * 60)
print("PHASE 3: Waiting for jobs to complete...")
print("=" * 60)

t0 = time.time()
for basin, job in jobs.items():
    print(f"\n  Waiting on {basin} ({BASIN_NAMES[basin]})...")
    job.wait_for_completion()
    print(f"    Status: {job.status}")
    if job.status == "Succeeded":
        job.retrieve_result()
        print(f"    Outputs: {len(job.outputs)} URL(s)")

elapsed = time.time() - t0
print(f"\nAll jobs finished in {elapsed:.0f}s")

# ============================================================
# Phase 4: Download outputs
# ============================================================
print("\n" + "=" * 60)
print("PHASE 4: Downloading outputs")
print("=" * 60)

multi_output_dir = WORK_DIR / "output_multi"
multi_output_dir.mkdir(exist_ok=True)

for basin, job in jobs.items():
    if job.status != "Succeeded":
        print(f"\n  {basin}: SKIPPED (status={job.status})")
        continue

    basin_out = multi_output_dir / basin
    basin_out.mkdir(exist_ok=True)

    for output_url in job.outputs:
        if not output_url.startswith("s3://"):
            continue
        bucket_name, prefix = parse_dps_s3_url(output_url)
        response = s3_ws.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
        for obj in response.get("Contents", []):
            key = obj["Key"]
            filename = key.split("/")[-1]
            if not filename:
                continue
            local_out = basin_out / filename
            s3_ws.download_file(bucket_name, key, str(local_out))

    files = list(basin_out.iterdir())
    print(f"  {basin} ({BASIN_NAMES[basin]}): {len(files)} file(s)")
    for f in files:
        print(f"    {f.name} ({f.stat().st_size / 1024:.1f} KB)")

print("\nDone!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import netCDF4
import numpy as np
from datetime import datetime, timedelta

print("=== Multi-Basin Output Visualization ===\n")

fig, axes = plt.subplots(len(BASINS), 1, figsize=(12, 4 * len(BASINS)), sharex=True)
if len(BASINS) == 1:
    axes = [axes]

colors = ["#1f77b4", "#2ca02c", "#d62728"]

for ax, basin, color in zip(axes, BASINS, colors):
    qout_file = multi_output_dir / basin / "Qout_maap.nc4"
    if not qout_file.exists():
        ax.text(0.5, 0.5, f"{basin}: No output file", transform=ax.transAxes,
                ha="center", va="center", fontsize=14)
        ax.set_ylabel(f"{basin}\n{BASIN_NAMES[basin]}")
        continue

    with netCDF4.Dataset(qout_file, "r") as ds:
        time_var = ds.variables["time"][:]
        time_units = ds.variables["time"].units
        qout = ds.variables["Qout"][:]
        rivid = ds.variables["rivid"][:]

        # Convert time to datetime
        ref_date = datetime(1970, 1, 1)
        dates = [ref_date + timedelta(seconds=float(t)) for t in time_var]

        # Find the reach with maximum mean discharge (the main stem outlet)
        mean_q = np.mean(qout, axis=0)
        main_reach_idx = np.argmax(mean_q)
        main_reach_id = rivid[main_reach_idx]

        # Also get 90th percentile reach for context
        p90_idx = np.argsort(mean_q)[int(0.9 * len(mean_q))]

        # Plot main stem
        ax.plot(dates, qout[:, main_reach_idx], color=color, linewidth=1.5,
                label=f"Main outlet (rivid {main_reach_id}, mean={mean_q[main_reach_idx]:.1f} m³/s)")
        ax.fill_between(dates, 0, qout[:, main_reach_idx], alpha=0.15, color=color)

        # Plot 90th percentile reach as dashed
        ax.plot(dates, qout[:, p90_idx], color=color, linewidth=0.8, linestyle="--", alpha=0.6,
                label=f"90th pctl reach (rivid {rivid[p90_idx]})")

        ax.set_ylabel("Discharge (m³/s)")
        ax.set_title(f"{basin} — {BASIN_NAMES[basin]} ({len(rivid):,} reaches)", fontsize=12)
        ax.legend(loc="upper right", fontsize=9)
        ax.grid(True, alpha=0.3)

        print(f"  {basin} ({BASIN_NAMES[basin]}):")
        print(f"    Reaches: {len(rivid):,}")
        print(f"    Time steps: {len(dates)}")
        print(f"    Main outlet rivid: {main_reach_id} (mean Q = {mean_q[main_reach_idx]:.1f} m³/s)")
        print(f"    Max instantaneous Q: {np.max(qout[:, main_reach_idx]):.1f} m³/s")

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
axes[-1].set_xlabel("Date")
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(str(multi_output_dir / "multi_basin_hydrographs.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {multi_output_dir}/multi_basin_hydrographs.png")